# MKWii RL Training Monitor
Run the cell below to start training and monitor episode rewards in real time.

In [ ]:
import subprocess, sys, os

RUNS_DIR = os.path.join(os.path.dirname(os.getcwd()), "runs")
tb_proc = subprocess.Popen(
    [sys.executable, "-m", "tensorboard.main", "--logdir", RUNS_DIR, "--port", "6006"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f"TensorBoard running at http://localhost:6006  (logdir: {RUNS_DIR})")
print("Run the cell below to start training. Stop this cell to shut down TensorBoard.")

TensorBoard running at http://localhost:6006  (logdir: c:\Users\Nicolas\LocalDocuments\HSLU\DSPRO2\runs)
Run the cell below to start training. Stop this cell to shut down TensorBoard.


In [5]:
import subprocess, sys, re, os
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import display
import ipywidgets as widgets
import json

import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

matplotlib.rcParams['figure.figsize'] = (12, 5)

PROJECT_ROOT = os.path.dirname(os.getcwd())
START_SCRIPT = os.path.join(PROJECT_ROOT, "scripts", "StartTraining.py")
STATE_FILE = os.path.join(PROJECT_ROOT, "scripts", "training_state.json")

try:
    with open(STATE_FILE) as f:
        episode_offset = json.load(f).get("episode_count", 0)
except Exception:
    episode_offset = 0

# Storage for episode data
p1_rewards = []
p2_rewards = []
p1_episodes = []
p2_episodes = []
episode_count = 0

PATTERN = re.compile(r'\[TrainingProcess\] P(\d) episode \d+ end\. stuck=(\w+) total_reward=([\-\d\.]+)')

plot_output = widgets.Output()
display(plot_output)

def update_plot():
    with plot_output:
        plot_output.clear_output(wait=True)
        fig, ax1 = plt.subplots(1, 1)

        if p1_rewards:
            ax1.plot(p1_episodes, p1_rewards, 'o-', color='#00E5FF', label='P1', linewidth=1.5, markersize=4)
        if p2_rewards:
            ax1.plot(p2_episodes, p2_rewards, 'o-', color='#FF6B6B', label='P2', linewidth=1.5, markersize=4)
        ax1.axhline(y=0, color='white', linestyle='--', alpha=0.3)
        ax1.set_title('Episode Total Reward', color='white')
        ax1.set_xlabel('Episode', color='white')
        ax1.set_ylabel('Total Reward', color='white')
        ax1.legend()
        ax1.set_facecolor('#1a1a2e')
        fig.patch.set_facecolor('#0f0f23')
        ax1.tick_params(colors='white')
        ax1.spines['bottom'].set_color('white')
        ax1.spines['left'].set_color('white')
        ax1.spines['top'].set_visible(False)
        ax1.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.show()

def parse_line(line):
    global episode_count
    line = line.strip()
    if not line:
        return
    print(line, flush=True)
    matches = PATTERN.findall(line)
    for m in matches:
        player = int(m[0])
        reward = float(m[2])
        if player == 1:
            p1_rewards.append(reward)
        else:
            p2_rewards.append(reward)

    new_count = min(len(p1_rewards), len(p2_rewards))
    if new_count > episode_count:
        for i in range(episode_count + 1, new_count + 1):
            p1_episodes.append(episode_offset + i)
            p2_episodes.append(episode_offset + i)
        episode_count = new_count
        update_plot()

print(f"Starting training from: {START_SCRIPT}")
proc = subprocess.Popen(
    [sys.executable, "-u", START_SCRIPT],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in proc.stdout:
        parse_line(line)
except KeyboardInterrupt:
    proc.terminate()
    print("Training stopped.")

proc.wait()
# Delete training_ready file if it exists
training_ready_path = os.path.join(PROJECT_ROOT, "training_ready.txt")
if os.path.exists(training_ready_path):
    os.remove(training_ready_path)
    print("Deleted training_ready.txt")
print("Training process exited.")

True
NVIDIA GeForce RTX 3060


Output()

Starting training from: c:\Users\Nicolas\LocalDocuments\HSLU\DSPRO2\scripts\StartTraining.py
[StartTraining] Emulation speed set to 200%.
[StartTraining] Launching TrainingProcess...
[StartTraining] Waiting for TrainingProcess to be ready...
[NeuralAgent] Loaded model from c:\Users\Nicolas\LocalDocuments\HSLU\DSPRO2\scripts\..\agent_model.pth
[TrainingProcess] Starting up...
[TrainingProcess] Ports bound, ready file written.
[TrainingProcess] Waiting for Dolphin...
[StartTraining] TrainingProcess ready.
[StartTraining] Launching Dolphin...
[StartTraining] All systems go.
[TrainingProcess] P1 connected.
[TrainingProcess] P2 connected.
[TrainingProcess] Dolphin window not ready, retry 1/20...
[TrainingProcess] Dolphin window not ready, retry 1/20...
[DolphinCapture] Player 1 ready.
[DolphinCapture] Player 2 ready.
[TrainingProcess] P1 episode 1200 end. stuck=True total_reward=-255.64
[TrainingProcess] P2 episode 1200 end. stuck=True total_reward=-147.91


[TrainingProcess] P1 episode 1201 end. stuck=True total_reward=-890.47
[TrainingProcess] P2 episode 1201 end. stuck=True total_reward=-654.35


[TrainingProcess] P1 episode 1202 end. stuck=True total_reward=126.49
[TrainingProcess] P2 episode 1202 end. stuck=True total_reward=70.55


...

[TrainingProcess] P1 episode 1645 end. stuck=True total_reward=-56.06
[TrainingProcess] P2 episode 1645 end. stuck=True total_reward=13.13


[TrainingProcess] P1 episode 1646 end. stuck=True total_reward=-16.49
[TrainingProcess] P2 episode 1646 end. stuck=True total_reward=62.84


[TrainingProcess] P1 episode 1647 end. stuck=True total_reward=-95.48
[TrainingProcess] P2 episode 1647 end. stuck=True total_reward=30.51


[TrainingProcess] P1 episode 1648 end. stuck=True total_reward=41.64
[TrainingProcess] P2 episode 1648 end. stuck=True total_reward=19.99


[DolphinCapture] Player 1 capture ended.
[DolphinCapture] Player 2 capture ended.
Exception in thread Thread-2 (player_loop):
Traceback (most recent call last):
File "C:\Users\Nicolas\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\threading.py", line 1075, in _bootstrap_inner
Exception in thread Thread-3 (player_loop):
Traceback (most recent call last):
File "C:\Users\Nicolas\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\threading.py", line 1075, in _bootstrap_inner
self.run()
File "C:\Users\Nicolas\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\threading.py", line 1012, in run
self._target(*self._args, **self._kwargs)
File "c:\Users\Nicolas\LocalDocuments\HSLU\DSPRO2\scripts\TrainingProcess.py", line 100, in player_loop
send_action(conn, action_idx)
File "c:\Users\Nicolas\LocalDocuments\HSLU\DSPRO2\scripts\TrainingProcess.py", line 24, in send_action
sock.sendall(struct.pack(">I", action_idx))
ConnectionResetError: [WinError 10054] Eine